In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/heart-failure-clinical-data/heart_failure_clinical_records_dataset.csv


## Data Loading

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# Load Dataset
data_path = "/kaggle/input/heart-failure-clinical-data/heart_failure_clinical_records_dataset.csv"
data = pd.read_csv(data_path)
data.head()


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


## Exploratory data analysis

In [ ]:
# ----------------------------
# 1. Basic Info About the Data
# ----------------------------
print("\n----- FIRST 5 ROWS -----")
print(data.head())

print("\n----- DATA INFO -----")
print(data.info())

print("\n----- SUMMARY STATISTICS -----")
print(data.describe())

# ----------------------------
# 2. Check for Missing Values
# ----------------------------
print("\n----- MISSING VALUES -----")
print(data.isnull().sum())

In [ ]:
# ----------------------------
# 3. Target Variable Distribution
# ----------------------------
plt.figure()

# Get value counts
counts = data['DEATH_EVENT'].value_counts()

# Plot with custom colors
counts.plot(kind='bar', color=['blue', 'red'])  

plt.title('Distribution of DEATH_EVENT')
plt.xlabel('DEATH_EVENT')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
# ----------------------------
# 4. Correlation Heatmap
# ----------------------------
corr = data.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
plt.imshow(corr, aspect='auto')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.colorbar()
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()


In [ ]:
# ----------------------------
# 5. Boxplots of Features vs Target
# ----------------------------
numeric_cols = data.select_dtypes(include='number').columns
numeric_cols = [col for col in numeric_cols if col != 'DEATH_EVENT']

for col in numeric_cols:
    plt.figure()
    data.boxplot(column=col, by='DEATH_EVENT')
    plt.title(f"{col} vs DEATH_EVENT")
    plt.suptitle("")  # Remove automatic title
    plt.xlabel("DEATH_EVENT")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()


In [ ]:
# ----------------------------
# 6. Histograms of All Features
# ----------------------------
data.hist(figsize=(12, 10))
plt.tight_layout()
plt.show()

# Building the Linear Regression Model
## Divide data into features and target

In [ ]:
# Target column
y = data["DEATH_EVENT"].values.reshape(-1, 1)
# Feature matrix (all numeric features except target)
X_raw = data.drop(columns=["DEATH_EVENT"]).values
# Number of samples & features
n, p = X_raw.shape
print("Samples:", n, " Features:", p)
print(y[:5])
print(X_raw[:5])

## Standardizing Features

In [ ]:
# Standardize Features
means = X_raw.mean(axis=0, keepdims=True)
stds  = X_raw.std(axis=0, keepdims=True)
stds[stds == 0] = 1.0

X = (X_raw - means) / stds   # standardized
print(X[:5])

## Creating bias column and adding it to our dataset

In [ ]:
# Create a column of ones with the same number of rows as our data
ones_column = np.ones((X.shape[0], 1))
# Stack the ones column with the original features
X_matrix = np.hstack((ones_column, X))

print("--- Processed X Matrix (First 5 rows) ---")
print(X_matrix[:5])
print(f"\nShape of X Matrix: {X_matrix.shape} (Rows, Columns)")

## 3. Implementing the Normal Equation

### Now we solve for $B$.

### The Formula: $B = (X^T X)^{-1} X^T Y$


### Step 1: Calculate X Transpose (X^T)

In [ ]:
X_matrix_T = X_matrix.T


print(f"Shape of X_T: {X_matrix_T.shape}")

### Step 2 : Calculate X Transpose times X (X^T X)
### This results in a square matrix (13x13 in our case)

In [ ]:
X_T_X = np.matmul(X_matrix_T, X_matrix)

print("\n--- Matrix (X^T * X) ---")
print(X_T_X)
print ("Shape of X_TX ",X_T_X.shape)

### Step 3: Calculate Inverse of (X^T X)^(-1) 

In [ ]:
try:
    X_T_X_Inv = np.linalg.inv(X_T_X)
    print("\n--- Inverse Matrix (X^T * X)^-1 ---")
    print(X_T_X_Inv)
except np.linalg.LinAlgError:
    print("Matrix is singulard cannot an be inverted.")
print("Shape of X_T_X Inverse : ", X_T_X_Inv.shape)

### Step 4: Calculate (X^T X)^(-1) X^T Y 

In [ ]:
# First we calculate the intermediate part: X^T * Y
X_T_Y = np.matmul(X_matrix_T, y)
# Now we calculate the final B vector
B = np.matmul(X_T_X_Inv, X_T_Y)

print("\n--- Calculated Coefficients (B Vector) ---")
print(B)

## 4. Interpreting the Results
# 
### The vector $B$ contains our coefficients $[b_0, b_1, b_2, b_3, b_4, b_5, b_6, b_7, b_8, b_9, b_10, b_11, b_12]$.
# 
### * $b_0$: Intercept (Bias)
### * $b_1$: Weight for Age
### * $b_2$: Weight for Anaemia
### * $b_3$: Weight for Creatinine Phosphokinase
### * $b_4$: Weight for Diabetes
### * $b_5$: Weight for Ejection Fraction
### * $b_6$: Weight for High Blood pressure
### * $b_7$: Weight for Platelets
### * $b_8$: Weight for Serum Creatinine
### * $b_9$: Weight for Serum Sodium
### * $b_10$: Weight for Sex
### * $b_11$: Weight for Smoking
### * $b_12$: Weight for Time

In [ ]:
b0 = B[0][0]
b1 = B[1][0]
b2 = B[2][0]
b3 = B[3][0]
b4 = B[4][0]
b5 = B[5][0]
b6 = B[6][0]
b7 = B[7][0]
b8 = B[8][0]
b9 = B[9][0]
b10 = B[10][0]
b11 = B[11][0]
b12 = B[12][0]

print("--- Final Model ---")
print(f"b0 (Intercept)    : {b0:.4f}")
print(f"b1 (age): {b1:.4f}")
print(f"b2 (anaemia)    : {b2:.4f}")
print(f"b3 (creatinine_phosphokinase)    : {b3:.4f}")
print(f"b4 (diabetes): {b4:.4f}")
print(f"b5 (ejection_fraction)    : {b5:.4f}")
print(f"b6 (high_blood_pressure)    : {b6:.4f}")
print(f"b7 (platelets): {b7:.4f}")
print(f"b8 (serum_creatinine)    : {b8:.4f}")
print(f"b9 (serum_sodium)    : {b9:.4f}")
print(f"b10 (sex): {b10:.4f}")
print(f"b11 (smoking)    : {b11:.4f}")
print(f"b12 (time)    : {b12:.4f}")
print("\nEquation:")
print(f"Survival event = {b0:.4f} + ({b1:.4f} * age) + ({b2:.4f} * anemia)+ ({b3:.4f} * creatinine phosphokinase)+ ({b4:.4f} * diabetes)+ ({b5:.4f} * ejection fraction)+ ({b6:.4f} * high blood pressure)+ ({b7:.4f} * platelets)+ ({b8:.4f} * serum creatinine)+ ({b9:.4f} * serum sodium)+ ({b10:.4f}  sex)+ ({b11:.4f} * smokimg)+ ({b12:.4f} * time)")


## 5. Verification
 
### Let's verify our manual matrix math against NumPy's built-in optimized solver `numpy.linalg.lstsq`.
### If our math is correct, the results should be identical.

### Using numpy's built-in least squares solver
### rcond=None ensures it handles machine precision warnings

In [ ]:
results_verification = np.linalg.lstsq(X_matrix, y, rcond=None)[0]

print("--- Verification Results (NumPy Built-in) ---")
print(results_verification)

print("\n--- Do they match? ---")
is_close = np.allclose(B, results_verification)
print(f"Match Status: {is_close}")


## 6. Testing the model
 


In [ ]:
# -------------------------------
# 1. Predict continuous DEATH_EVENT for all samples
# -------------------------------
y_pred_continuous = np.matmul(X_matrix, B)

# -------------------------------
# 2. Convert to binary classification using 0.5 threshold
# -------------------------------
y_pred_class = (y_pred_continuous >= 0.5).astype(int)

# Check first 10 predictions
print("\nFirst 10 predicted classes (0=alive, 1=death):")
print(y_pred_class[:10])

# -------------------------------
# 3. Example prediction for a new patient
# -------------------------------
# New patient data (replace with actual values)
# Format: age, anaemia, creatinine_phosphokinase, diabetes, ejection_fraction, high_blood_pressure,
# platelets, serum_creatinine, serum_sodium, sex, smoking, time
new_patient_raw = np.array([[75, 0, 582, 0, 20, 1, 265000, 1.9, 130, 1, 0, 4]])

# Standardize using training data mean/std
new_patient_std = (new_patient_raw - means) / stds

# Add intercept
new_patient_matrix = np.hstack([np.ones((1, 1)), new_patient_std])

# Continuous prediction
new_patient_pred_cont = np.matmul(new_patient_matrix, B)

# Binary prediction
new_patient_pred_class = (new_patient_pred_cont >= 0.5).astype(int)

print("\nExample Prediction for New Patient:")
print("Continuous prediction:", new_patient_pred_cont[0][0])
print("Predicted DEATH_EVENT class (0=alive, 1=death):", new_patient_pred_class[0][0])


## Model Metrics Evaluation

In [ ]:
# 1. Generate predictions for the WHOLE dataset using our matrix math
# Formula: Y_pred = X_matrix * B
Y_pred_all = np.matmul(X_matrix, B)

# --- Metric 1: RMSE (Root Mean Squared Error) ---
# Calculate the average squared difference between Actual and Predicted
mse = np.mean((y - Y_pred_all) ** 2)
rmse = np.sqrt(mse)

# --- Metric 2: R-Squared (Coefficient of Determination) ---
# Formula: 1 - (Sum of Squared Residuals / Total Sum of Squares)
ss_residual = np.sum((y - Y_pred_all) ** 2)
ss_total = np.sum((y - np.mean(y)) ** 2)
r2 = 1 - (ss_residual / ss_total)

print("--- Model Metrics (Calculated from Scratch) ---")
print(f"R-Squared Score : {r2:.4f} (The model explains {r2*100:.1f}% of the variance)")

print('MSE Error')
print(mse)
print(f"RMSE Error      : {rmse:.4f} (On average, the model is off by {rmse:.4f})")

# Note: Accuracy/Precision are not used here because this is Regression (predicting a number),
# not Classification (predicting a category like Yes/No).